# Online Retail — End-to-End Analysis
**Dataset:** UCI Online Retail Dataset  
**Goal:** Predict future customer spend using past transaction behavior (Customer Lifetime Value)  
**Approach:** Feature engineering → Temporal train/test split → Linear Regression vs Random Forest

### Key design decision
Features are built from the **first 6 months** of transaction history (before June 2011).  
The target is **future spend in the last 6 months** (after June 2011).  
This avoids data leakage — the model genuinely cannot see the future when training.

---

## 0. Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.inspection import permutation_importance

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

---
## 1. Load & Clean Data

In [ ]:
df = pd.read_excel('Online Retail.xlsx', dtype={'CustomerID': str})

# Remove cancellations (InvoiceNo starting with 'C')
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Drop missing CustomerID and Description
df = df.dropna(subset=['CustomerID', 'Description'])

# Remove negative or zero Quantity and UnitPrice
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Create Revenue column
df['Revenue']     = df['Quantity'] * df['UnitPrice']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['YearMonth']   = df['InvoiceDate'].dt.to_period('M')

print(f'Clean dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Date range: {df["InvoiceDate"].min().date()} to {df["InvoiceDate"].max().date()}')
df.head()

---
## 2. Exploratory Plots

In [ ]:
# Monthly revenue over time
monthly_revenue = df.groupby('YearMonth')['Revenue'].sum().reset_index()
monthly_revenue['YearMonth'] = monthly_revenue['YearMonth'].astype(str)

plt.figure(figsize=(14, 5))
plt.plot(monthly_revenue['YearMonth'], monthly_revenue['Revenue'],
         marker='o', color='steelblue', linewidth=2)
plt.axvline(x='2011-06', color='red', linestyle='--',
            linewidth=1.5, label='Train/test cutoff (Jun 2011)')
plt.title('Monthly Revenue Over Time')
plt.xlabel('Month')
plt.ylabel('Revenue (£)')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 products by revenue
top_products = (df.groupby('Description')['Revenue']
                  .sum().sort_values(ascending=False)
                  .head(10).reset_index())

plt.figure(figsize=(12, 5))
sns.barplot(data=top_products, x='Revenue', y='Description', palette='Blues_r')
plt.title('Top 10 Products by Revenue')
plt.xlabel('Total Revenue (£)')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 countries by revenue (excl. UK)
top_countries = (df[df['Country'] != 'United Kingdom']
                   .groupby('Country')['Revenue']
                   .sum().sort_values(ascending=False)
                   .head(10).reset_index())

plt.figure(figsize=(12, 5))
sns.barplot(data=top_countries, x='Revenue', y='Country', palette='Greens_r')
plt.title('Top 10 Countries by Revenue (Excluding UK)')
plt.xlabel('Total Revenue (£)')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Revenue distribution per transaction
revenue_cap = df['Revenue'].quantile(0.95)
df_capped   = df[df['Revenue'] <= revenue_cap]

plt.figure(figsize=(12, 5))
sns.histplot(df_capped['Revenue'], bins=80, color='coral', kde=True)
plt.title('Revenue Distribution per Transaction (up to 95th percentile)')
plt.xlabel('Revenue (£)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

---
## 3. Temporal Train/Test Split

To avoid data leakage, we use a **temporal split**:
- **Past (before Jun 2011)** → used to build features
- **Future (Jun 2011 onwards)** → used as the prediction target

Only customers who appear in **both** periods are kept — we need past behavior to predict and future spend to evaluate against.

In [ ]:
cutoff_date = pd.Timestamp('2011-06-01')

past   = df[df['InvoiceDate'] <  cutoff_date]
future = df[df['InvoiceDate'] >= cutoff_date]

print(f'Past transactions   : {len(past):,}')
print(f'Future transactions : {len(future):,}')
print(f'Past customers      : {past["CustomerID"].nunique():,}')
print(f'Future customers    : {future["CustomerID"].nunique():,}')

---
## 4. Feature Engineering (from Past Only)

In [ ]:
# Build all features from PAST transactions only
past_features = past.groupby('CustomerID').agg(
    Recency         = ('InvoiceDate', lambda x: (cutoff_date - x.max()).days),
    Frequency       = ('InvoiceNo',   'nunique'),
    Monetary_Past   = ('Revenue',     'sum'),
    Unique_Products = ('StockCode',   'nunique'),
    Max_Order       = ('Revenue',     'max'),
    Std_Order       = ('Revenue',     'std'),
    Avg_Items       = ('Quantity',    'mean'),
).reset_index()
past_features['Std_Order'] = past_features['Std_Order'].fillna(0)

# Build TARGET from FUTURE transactions only
future_target = (future.groupby('CustomerID')['Revenue']
                       .sum()
                       .reset_index()
                       .rename(columns={'Revenue': 'Future_Monetary'}))

# Keep only customers present in BOTH periods
model_df = past_features.merge(future_target, on='CustomerID', how='inner')
model_df = model_df[model_df['Future_Monetary'] > 0].copy()

print(f'Customers in both periods: {model_df.shape[0]:,}')
print(f'\nFuture_Monetary stats:')
print(model_df['Future_Monetary'].describe())

In [ ]:
# Log transform all skewed features
log_cols = ['Recency', 'Frequency', 'Monetary_Past', 'Unique_Products',
            'Max_Order', 'Std_Order', 'Avg_Items', 'Future_Monetary']

for col in log_cols:
    model_df[f'Log_{col}'] = np.log1p(model_df[col])

# Plot distributions before and after log transform
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
cols_to_plot = ['Recency', 'Frequency', 'Monetary_Past', 'Future_Monetary']

for i, col in enumerate(cols_to_plot):
    axes[0, i].hist(model_df[col],         bins=50, color='coral',     edgecolor='white')
    axes[0, i].set_title(f'{col} — original')
    axes[1, i].hist(model_df[f'Log_{col}'], bins=50, color='steelblue', edgecolor='white')
    axes[1, i].set_title(f'{col} — log')

plt.tight_layout()
plt.show()

---
## 5. Correlation Analysis

In [ ]:
feature_cols = ['Log_Recency', 'Log_Frequency', 'Log_Monetary_Past',
                'Log_Unique_Products', 'Log_Max_Order',
                'Log_Std_Order', 'Log_Avg_Items']

corr = (model_df[feature_cols + ['Log_Future_Monetary']]
        .corr()['Log_Future_Monetary']
        .drop('Log_Future_Monetary')
        .sort_values(key=abs, ascending=False)
        .reset_index())
corr.columns = ['Feature', 'Correlation']

print(corr.to_string(index=False))

plt.figure(figsize=(9, 5))
colors = ['steelblue' if c > 0 else 'coral' for c in corr['Correlation']]
plt.barh(corr['Feature'], corr['Correlation'],
         color=colors, edgecolor='white', height=0.6)
plt.axvline(0, color='black', linewidth=1)
plt.title('Feature correlation with Log(Future Monetary)')
plt.xlabel('Pearson correlation coefficient')
plt.tight_layout()
plt.show()

---
## 6. Model Training & Evaluation

**Target:** `Log_Future_Monetary` — log of customer spend in the 6 months after the cutoff date  
**Features:** All derived from the 6 months *before* the cutoff date  
**Split:** Random 80/20 within the eligible customer pool

In [ ]:
features_final = [
    'Log_Recency',
    'Log_Frequency',
    'Log_Monetary_Past',
    'Log_Unique_Products',
    'Log_Max_Order',
    'Log_Std_Order',
    'Log_Avg_Items'
]

X = model_df[features_final]
y = model_df['Log_Future_Monetary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {X_train.shape[0]:,}')
print(f'Testing samples  : {X_test.shape[0]:,}')

In [ ]:
# --- Linear Regression ---
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

r2_lr   = r2_score(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr  = np.mean(np.abs(y_test - y_pred_lr))

print('--- Linear Regression Coefficients ---')
for name, coef in zip(features_final, lr.coef_):
    print(f'  {name:22}: {coef:.4f}')
print(f'  {"Intercept":22}: {lr.intercept_:.4f}')
print(f'\n  R²   : {r2_lr:.4f}')
print(f'  RMSE : {rmse_lr:.4f}')
print(f'  MAE  : {mae_lr:.4f}')

In [ ]:
# --- Random Forest ---
features_rf = ['Recency', 'Frequency', 'Monetary_Past', 'Unique_Products',
               'Max_Order', 'Std_Order', 'Avg_Items']

X_rf = model_df[features_rf]
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_rf, y, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(
    n_estimators=200, max_depth=10,
    min_samples_leaf=5, random_state=42, n_jobs=-1
)
rf.fit(X_train_rf, y_train_rf)
y_pred_rf = rf.predict(X_test_rf)

r2_rf   = r2_score(y_test_rf, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test_rf, y_pred_rf))
mae_rf  = np.mean(np.abs(y_test_rf - y_pred_rf))

print(f'Random Forest  R²={r2_rf:.4f}  RMSE={rmse_rf:.4f}  MAE={mae_rf:.4f}')

---
## 7. Diagnostic Plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for row, (y_pred, y_te, label, color) in enumerate([
    (y_pred_lr, y_test,    'Linear Regression', 'steelblue'),
    (y_pred_rf, y_test_rf, 'Random Forest',     'mediumseagreen')
]):
    residuals = y_te - y_pred

    # Actual vs Predicted
    axes[row, 0].scatter(y_te, y_pred, alpha=0.3, color=color, s=10)
    axes[row, 0].plot([y_te.min(), y_te.max()],
                      [y_te.min(), y_te.max()],
                      color='red', linewidth=2, label='Perfect prediction')
    axes[row, 0].set_title(f'Actual vs Predicted — {label}')
    axes[row, 0].set_xlabel('Actual Log(Future Monetary)')
    axes[row, 0].set_ylabel('Predicted Log(Future Monetary)')
    axes[row, 0].legend()

    # Residuals distribution
    axes[row, 1].hist(residuals, bins=60, color='coral', edgecolor='white')
    axes[row, 1].axvline(0, color='black', linestyle='--', linewidth=1.5)
    axes[row, 1].set_title(f'Residuals — {label}')
    axes[row, 1].set_xlabel('Residual')
    axes[row, 1].set_ylabel('Count')

    # Permutation importance (Linear only — RF uses built-in)
    if row == 0:
        perm = permutation_importance(lr, X_test, y_test,
                                      n_repeats=30, random_state=42, scoring='r2')
        perm_df = pd.DataFrame({
            'Feature'   : features_final,
            'Importance': perm.importances_mean
        }).sort_values('Importance', ascending=False)
        axes[row, 2].barh(perm_df['Feature'], perm_df['Importance'],
                          color='steelblue', edgecolor='white', height=0.5)
        axes[row, 2].set_title('Permutation importance — Linear Regression')
    else:
        imp_df = pd.DataFrame({
            'Feature'   : features_rf,
            'Importance': rf.feature_importances_
        }).sort_values('Importance', ascending=False)
        axes[row, 2].barh(imp_df['Feature'], imp_df['Importance'],
                          color='mediumseagreen', edgecolor='white', height=0.5)
        axes[row, 2].set_title('Feature importance — Random Forest')

    axes[row, 2].axvline(0, color='black', linewidth=1)
    axes[row, 2].set_xlabel('Importance')

plt.tight_layout()
plt.show()

---
## 8. Final Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model'   : ['Linear Regression', 'Random Forest'],
    'Features': [len(features_final), len(features_rf)],
    'R²'      : [r2_lr,   r2_rf],
    'RMSE'    : [rmse_lr, rmse_rf],
    'MAE'     : [mae_lr,  mae_rf]
})
print(results.to_string(index=False))

colors = ['steelblue', 'mediumseagreen']
plt.figure(figsize=(8, 5))
bars = plt.bar(results['Model'], results['R²'],
               color=colors, edgecolor='white', width=0.4)
plt.axhline(0.5, color='black', linestyle='--', linewidth=1,
            label='0.50 reference')
plt.ylim(0, 0.8)
plt.title('R² Comparison — Predicting Future Customer Spend')
plt.ylabel('R²')
for bar, score in zip(bars, results['R²']):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.01,
             f'{score:.4f}', ha='center', fontsize=12, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

---
## 9. Key Findings & Lessons Learned

### Results

| Model | R² | RMSE | MAE |
|---|---|---|---|
| Linear Regression | 0.52 | 0.89 | — |
| Random Forest | 0.50 | 0.91 | — |

### What drives future customer spend?
1. **Past monetary spend** — how much they spent before is the strongest signal of future spend
2. **Frequency** — customers who ordered more often are more likely to continue
3. **Product variety** — customers who explored more products are more engaged
4. **Recency** — customers who bought recently are more likely to buy again

### Why Linear Regression matched Random Forest
- Only 1,927 eligible customers — too small for Random Forest to learn complex patterns
- After log transformation the relationship between past and future spend is mostly linear
- Random Forest slightly overfits on this dataset size

### The most important lesson — data leakage
Early versions of this analysis achieved R² of 0.95–0.98 by predicting **total spend from features derived from the same transactions**. This is data leakage:
- `Avg_Order_Value` equals `Monetary` for single-order customers (34% of dataset)
- Order-level aggregates (max, min, std, median) collectively reconstruct `Monetary` mathematically
- A high R² means nothing if your features are derived from the same data as your target

The correct approach uses a **temporal split** — features from the past, target from the future. The resulting R² of 0.52 is a genuinely trustworthy result for predicting future customer behavior.

### What would improve the model further?
- More customer history (this dataset only spans ~13 months)
- Product category features — what types of items they buy
- Seasonality features — whether their purchases cluster around holidays
- A longer future window — predicting 12 months instead of 6